In [13]:
import cv2
import base64
from pyzbar.pyzbar import decode

received_chunks = {}
total_chunks = None

# cap = cv2.VideoCapture(0)
cap = cv2.VideoCapture("./qr_transfer.gif")
print("Point your camera at the looping GIF. Press 'q' to quit early.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    decoded_objects = decode(frame)
    
    for obj in decoded_objects:
        try:
            # Safely decode the QR data as a standard text string
            payload_str = obj.data.decode('utf-8')
            
            # Split the header from the Base64 data
            header, b64_data = payload_str.split('|', 1)
            idx_str, total_str = header.split('/')
            
            idx = int(idx_str)
            total = int(total_str)
            
            if total_chunks is None:
                total_chunks = total
                
            if idx not in received_chunks:
                received_chunks[idx] = b64_data
                progress = (len(received_chunks) / total_chunks) * 100
                print(f"Caught {idx}/{total_chunks} ({progress:.1f}%)")
                
        except (ValueError, UnicodeDecodeError):
            # Ignore malformed or partial reads
            pass

    # UI Overlay
    display_text = f"Chunks: {len(received_chunks)} / {total_chunks if total_chunks else 'Waiting...'}"
    cv2.putText(frame, display_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.imshow("Optical Data Receiver", frame)

    if total_chunks and len(received_chunks) == total_chunks:
        print("\nAll chunks received successfully!")
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# --- Reassembly ---
if total_chunks and len(received_chunks) == total_chunks:
    print("Reassembling file from Base64...")
    
    # 1. Stitch the Base64 strings back together in order
    full_base64_string = "".join([received_chunks[i] for i in range(1, total_chunks + 1)])
    
    # 2. Decode back into raw binary file data
    try:
        file_bytes = base64.b64decode(full_base64_string)
        
        output_filename = "received_src.jpg"
        with open(output_filename, "wb") as f:
            f.write(file_bytes)
            
        print(f"Success! Uncorrupted file saved as: {output_filename}")
        
    except Exception as e:
        print(f"Error decoding Base64 data: {e}")
else:
    print(f"Transfer incomplete. Got {len(received_chunks)} out of {total_chunks} chunks.")

Point your camera at the looping GIF. Press 'q' to quit early.
Caught 1/176 (0.6%)
Caught 2/176 (1.1%)
Caught 3/176 (1.7%)
Caught 4/176 (2.3%)
Caught 5/176 (2.8%)
Caught 6/176 (3.4%)
Caught 7/176 (4.0%)
Caught 8/176 (4.5%)
Caught 9/176 (5.1%)
Caught 10/176 (5.7%)
Caught 11/176 (6.2%)
Caught 12/176 (6.8%)
Caught 13/176 (7.4%)
Caught 14/176 (8.0%)
Caught 15/176 (8.5%)
Caught 16/176 (9.1%)
Caught 17/176 (9.7%)
Caught 18/176 (10.2%)
Caught 19/176 (10.8%)
Caught 20/176 (11.4%)
Caught 21/176 (11.9%)
Caught 22/176 (12.5%)
Caught 23/176 (13.1%)
Caught 24/176 (13.6%)
Caught 25/176 (14.2%)
Caught 26/176 (14.8%)
Caught 27/176 (15.3%)
Caught 28/176 (15.9%)
Caught 29/176 (16.5%)
Caught 30/176 (17.0%)
Caught 31/176 (17.6%)
Caught 32/176 (18.2%)
Caught 33/176 (18.8%)
Caught 34/176 (19.3%)
Caught 35/176 (19.9%)
Caught 36/176 (20.5%)
Caught 37/176 (21.0%)
Caught 38/176 (21.6%)
Caught 39/176 (22.2%)
Caught 40/176 (22.7%)
Caught 41/176 (23.3%)
Caught 42/176 (23.9%)
Caught 43/176 (24.4%)
Caught 44/176 (25

In [22]:
import cv2
import base64
from pyzbar.pyzbar import decode

received_chunks = {}
total_chunks = None

cap = cv2.VideoCapture(0)
# cap = cv2.VideoCapture("./qr_transfer.gif")
print("Point your camera at the looping GIF. Press 'q' to quit early.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    decoded_objects = decode(frame)

    for obj in decoded_objects:
        try:
            # Safely decode the QR data as a standard text string
            payload_str = obj.data.decode('utf-8')

            # Split the header from the Base64 data
            header, b64_data = payload_str.split('|', 1)
            idx_str, total_str = header.split('/')

            idx = int(idx_str)
            total = int(total_str)

            if total_chunks is None:
                total_chunks = total

            if idx not in received_chunks:
                # Decode THIS chunk now, so its own '=' padding stays at its own end
                received_chunks[idx] = base64.b64decode(b64_data)
                progress = (len(received_chunks) / total_chunks) * 100
                print(f"Caught {idx}/{total_chunks} ({progress:.1f}%)")

        except ValueError:
            # Covers bad UTF-8, bad int(), bad split, and bad Base64
            pass

    # UI Overlay
    display_text = f"Chunks: {len(received_chunks)} / {total_chunks if total_chunks else 'Waiting...'}"
    cv2.putText(frame, display_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
    cv2.imshow("Optical Data Receiver", frame)

    if total_chunks and len(received_chunks) == total_chunks:
        print("\nAll chunks received successfully!")
        break

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# --- Reassembly ---
if total_chunks and len(received_chunks) == total_chunks:
    print("Reassembling file...")

    # Chunks are already raw bytes, so just join them in order
    file_bytes = b"".join(received_chunks[i] for i in range(1, total_chunks + 1))

    output_filename = "received_src.jpg"
    with open(output_filename, "wb") as f:
        f.write(file_bytes)

    print(f"Saved {output_filename} ({len(file_bytes)} bytes)")
else:
    missing = [i for i in range(1, (total_chunks or 0) + 1) if i not in received_chunks]
    print(f"Transfer incomplete. Got {len(received_chunks)} out of {total_chunks} chunks. Missing: {missing}")

Point your camera at the looping GIF. Press 'q' to quit early.
Caught 39/176 (0.6%)
Caught 47/176 (1.1%)
Caught 125/176 (1.7%)
Caught 126/176 (2.3%)
Caught 127/176 (2.8%)
Caught 128/176 (3.4%)
Caught 131/176 (4.0%)
Caught 132/176 (4.5%)
Caught 133/176 (5.1%)
Caught 134/176 (5.7%)
Caught 135/176 (6.2%)
Caught 136/176 (6.8%)
Caught 137/176 (7.4%)
Caught 139/176 (8.0%)
Caught 140/176 (8.5%)
Caught 141/176 (9.1%)
Caught 143/176 (9.7%)
Caught 145/176 (10.2%)
Caught 146/176 (10.8%)
Caught 147/176 (11.4%)
Caught 148/176 (11.9%)
Caught 149/176 (12.5%)
Caught 150/176 (13.1%)
Caught 151/176 (13.6%)
Caught 152/176 (14.2%)
Caught 153/176 (14.8%)
Caught 154/176 (15.3%)
Caught 155/176 (15.9%)
Caught 159/176 (16.5%)
Caught 160/176 (17.0%)
Caught 161/176 (17.6%)
Caught 163/176 (18.2%)
Caught 167/176 (18.8%)
Caught 168/176 (19.3%)
Caught 170/176 (19.9%)
Caught 172/176 (20.5%)
Caught 173/176 (21.0%)
Caught 176/176 (21.6%)
Caught 1/176 (22.2%)
Caught 12/176 (22.7%)
Caught 16/176 (23.3%)
Caught 21/176 (23